# 03 · Modelo estrella de ventas

Este cuaderno construye `dim_tiempo`, `dim_mercado`, `dim_cliente`, `dim_producto` y `fact_venta` a partir de las tablas limpias del mismo ETL que respalda el cuaderno 02. La auditoría de calidad se ejecuta en un bloque independiente, fuera del modelado dimensional.

**Grano de `fact_venta`:** una fila por `venta_id`, que identifica el registro. El hecho guarda claves foráneas hacia las dimensiones y medidas numéricas, sin atributos descriptivos ni flags. `dim_producto` tiene una fila por `producto_id` y `anada`, identificada mediante `producto_anada_key`; allí se almacenan `litros_por_caja` y `costo_caja_clp`. `dim_mercado` incluye `moneda_facturacion`, inferida una sola vez desde las monedas informadas del mercado, después de comprobar que cada mercado tenga una sola.

Las fechas de venta y embarque son claves de rol hacia `dim_tiempo`. La conversión a CLP usa `moneda_facturacion` y el dólar observado diario de Mindicador, completado con el último valor publicado para fines de semana y feriados. Si no hay conexión, `read_exchange_rate` reutiliza su caché local; si tampoco existe, hay que ejecutar antes la ingesta con tipo de cambio.

Las celdas muestran el diccionario explícito de hemisferios, las inconsistencias de asignación cliente-mercado y los controles de moneda, grano, tipos e integridad referencial. El modelo queda en memoria como DataFrames y no publica archivos.

In [22]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))

from vina_vallehondo_bi.etl.processed import (
    clean_costs,
    clean_customers,
    clean_exchange_rate,
    clean_markets,
    clean_sales,
    clean_shipments,
)
from vina_vallehondo_bi.etl.staging import (
    read_costs,
    read_dimensions,
    read_exchange_rate,
    read_sales,
    read_shipments,
)

stg_ventas = read_sales(ROOT)
dims_fuente = read_dimensions(ROOT)
stg_costos = read_costs(ROOT)
stg_embarques = read_shipments(ROOT)

ventas = clean_sales(stg_ventas)
mercados = clean_markets(
    dims_fuente["stg_mercados"], dims_fuente["stg_zonas"], stg_ventas
)
clientes = clean_customers(dims_fuente["stg_clientes"])
productos = clean_costs(stg_costos, dims_fuente["stg_cepas"])
embarques = clean_shipments(stg_embarques, ventas)
monedas = dims_fuente["stg_monedas"].copy()

print(
    f"ventas limpias: {len(ventas):,} | mercados: {len(mercados)} | "
    f"clientes: {len(clientes)} | producto-añada: {len(productos)}"
)
print(f"embarques: {len(embarques):,} líneas limpias")

ventas limpias: 13,830 | mercados: 11 | clientes: 26 | producto-añada: 96
embarques: 10,715 líneas limpias


In [29]:
HEMISFERIO_POR_PAIS = {
    "Chile": "Sur",
    "Brasil": "Sur",
    "México": "Norte",
    "Estados Unidos": "Norte",
    "Canadá": "Norte",
    "Reino Unido": "Norte",
    "Alemania": "Norte",
    "Países Bajos": "Norte",
    "China": "Norte",
    "Japón": "Norte",
    "Corea del Sur": "Norte",
}

paises_sin_hemisferio = sorted(set(mercados["pais"]) - set(HEMISFERIO_POR_PAIS))
assert not paises_sin_hemisferio, f"Falta clasificar países: {paises_sin_hemisferio}"
print("Diccionario explícito de hemisferios (revisar):")
print(pd.Series(HEMISFERIO_POR_PAIS, name="hemisferio").rename_axis("pais").to_frame().sort_index())

ventas_con_moneda = stg_ventas.dropna(subset=["moneda_id"])
monedas_por_mercado = ventas_con_moneda.groupby("mercado_id")["moneda_id"].nunique()
mercados_con_varias_monedas = monedas_por_mercado[monedas_por_mercado > 1]
assert mercados_con_varias_monedas.empty, (
    "No se puede inferir moneda por mercado; más de una moneda informada: "
    f"{mercados_con_varias_monedas.to_dict()}"
)
moneda_id_por_mercado = ventas_con_moneda.groupby("mercado_id")["moneda_id"].first()
mercados_sin_moneda_informada = sorted(set(mercados["mercado_id"]) - set(moneda_id_por_mercado.index))
assert not mercados_sin_moneda_informada, (
    f"Mercados sin ninguna moneda informada: {mercados_sin_moneda_informada}"
)
codigo_por_moneda = monedas.set_index("moneda_id")["moneda"].str.upper()

dim_mercado = mercados[["mercado_id", "pais", "zona", "canal"]].copy()
dim_mercado["hemisferio"] = dim_mercado["pais"].map(HEMISFERIO_POR_PAIS)
dim_mercado["tratado_libre_comercio"] = np.where(
    dim_mercado["pais"].eq("Chile"), "No aplica", "Sí"
)
dim_mercado["moneda_facturacion"] = (
    dim_mercado["mercado_id"].map(moneda_id_por_mercado).map(codigo_por_moneda)
)
assert dim_mercado["moneda_facturacion"].notna().all()
dim_mercado = dim_mercado[
    [
        "mercado_id",
        "pais",
        "zona",
        "hemisferio",
        "tratado_libre_comercio",
        "moneda_facturacion",
    ]
]
print("\nMoneda informada por mercado (debe ser única por mercado):")
print(dim_mercado[["mercado_id", "pais", "moneda_facturacion"]].to_string(index=False))

dim_cliente = clientes[["cliente_id", "nombre", "tipo"]].drop_duplicates("cliente_id").copy()
mercados_por_cliente = ventas.groupby("cliente_id")["mercado_id"].nunique()
clientes_multi_mercado = mercados_por_cliente[mercados_por_cliente > 1]
print(f"\nClientes que aparecen en más de un mercado en ventas: {len(clientes_multi_mercado)}")
if not clientes_multi_mercado.empty:
    print(
        ventas[ventas["cliente_id"].isin(clientes_multi_mercado.index)]
        .merge(dim_cliente, on="cliente_id", how="left")
        .groupby(["cliente_id", "nombre", "mercado_id"], as_index=False)
        .size()
    )

mercado_maestro = clientes.set_index("cliente_id")["mercado_id"]
mercado_cliente = ventas.groupby("cliente_id")["mercado_id"].agg(
    lambda values: tuple(sorted(values.dropna().unique()))
).rename("mercados_en_ventas").to_frame()
mercado_cliente["mercado_maestro"] = mercado_maestro.reindex(mercado_cliente.index)
mask_mercado_distinto = mercado_cliente.apply(
    lambda row: len(row["mercados_en_ventas"]) != 1
    or row["mercados_en_ventas"][0] != row["mercado_maestro"],
    axis=1,
)
clientes_distintos_maestro = mercado_cliente[mask_mercado_distinto]
print(f"Clientes cuyo mercado en ventas difiere del maestro: {len(clientes_distintos_maestro)}")
if not clientes_distintos_maestro.empty:
    print(clientes_distintos_maestro)

dim_producto = productos[
    [
        "producto_id",
        "anada",
        "cepa",
        "tier",
        "formato",
        "litros_por_caja",
        "costo_caja_clp",
    ]
].copy()
dim_producto = dim_producto.sort_values(["producto_id", "anada"]).reset_index(drop=True)
dim_producto.insert(0, "producto_anada_key", np.arange(1, len(dim_producto) + 1))

embarques_fact = (
    embarques[embarques["fecha_embarque_confiable"]]
    .sort_values(["fecha_embarque", "embarque_orden", "linea_orden"])
    .drop_duplicates("venta_id", keep="first")[["venta_id", "fecha_embarque"]]
)
fechas = pd.concat(
    [ventas["fecha"], embarques_fact["fecha_embarque"]], ignore_index=True
).dropna()
calendario = pd.date_range(fechas.min().normalize(), fechas.max().normalize(), freq="D")
dim_tiempo = pd.DataFrame({"fecha": calendario})
dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["trimestre"] = "T" + dim_tiempo["fecha"].dt.quarter.astype(str)
dim_tiempo["semestre"] = "S" + (((dim_tiempo["mes"] - 1) // 6) + 1).astype(str)
dim_tiempo["estacion_hemisferio_sur"] = dim_tiempo["mes"].map(
    {
        12: "Verano", 1: "Verano", 2: "Verano",
        3: "Otoño", 4: "Otoño", 5: "Otoño",
        6: "Invierno", 7: "Invierno", 8: "Invierno",
        9: "Primavera", 10: "Primavera", 11: "Primavera",
    }
)

print(
    f"\nDimensiones: tiempo={len(dim_tiempo):,} días | mercado={len(dim_mercado)} | "
    f"cliente={len(dim_cliente)} | producto-añada={len(dim_producto)}"
)
print(
    "Fechas confiables elegidas: "
    f"{len(embarques_fact):,} ventas; líneas confiables duplicadas por venta: "
    f"{int(embarques.loc[embarques['fecha_embarque_confiable'], 'venta_id'].duplicated().sum()):,}"
)

Diccionario explícito de hemisferios (revisar):
               hemisferio
pais                     
Alemania            Norte
Brasil                Sur
Canadá              Norte
Chile                 Sur
China               Norte
Corea del Sur       Norte
Estados Unidos      Norte
Japón               Norte
México              Norte
Países Bajos        Norte
Reino Unido         Norte

Moneda informada por mercado (debe ser única por mercado):
 mercado_id           pais moneda_facturacion
          1          Chile                CLP
          2         Brasil                USD
          3         México                USD
          4 Estados Unidos                USD
          5         Canadá                USD
          6    Reino Unido                USD
          7       Alemania                USD
          8   Países Bajos                USD
          9          China                USD
         10          Japón                USD
         11  Corea del Sur                USD

C

In [33]:
cepas_tipo = dims_fuente["stg_cepas"][["cepa", "tipo"]].drop_duplicates("cepa")
dim_producto = dim_producto.merge(
    cepas_tipo,
    on="cepa",
    how="left",
    validate="many_to_one",
)
assert dim_producto["tipo"].notna().all(), "Hay productos sin tipo de vino en el maestro de cepas."
dim_producto = dim_producto[
    [
        "producto_anada_key",
        "producto_id",
        "anada",
        "cepa",
        "tipo",
        "tier",
        "formato",
        "litros_por_caja",
        "costo_caja_clp",
    ]
]
print("Tipo de vino unido desde el maestro de cepas:")
print(dim_producto.groupby(["cepa", "tipo"]).size().rename("producto-añada"))

Tipo de vino unido desde el maestro de cepas:
cepa                tipo  
Cabernet Sauvignon  Tinto     16
Carménère           Tinto     16
Chardonnay          Blanco    16
Merlot              Tinto     12
Pinot Noir          Tinto     12
Sauvignon Blanc     Blanco    12
Syrah               Tinto     12
Name: producto-añada, dtype: int64


In [25]:
tasas_staging = read_exchange_rate(ROOT)
tipo_cambio = clean_exchange_rate(tasas_staging)
tipo_cambio = tipo_cambio.dropna(subset=["dolar_observado"]).drop_duplicates("fecha")
tasa_por_fecha = tipo_cambio.set_index("fecha")["dolar_observado"]

print(
    f"Dólar observado diario: {len(tipo_cambio):,} días con tasa, "
    f"desde {tipo_cambio['fecha'].min():%Y-%m-%d} hasta {tipo_cambio['fecha'].max():%Y-%m-%d}; "
    f"{int(tipo_cambio['publicado'].sum()):,} fechas publicadas y "
    f"{int((~tipo_cambio['publicado']).sum()):,} completadas con el último valor publicado."
)

Aviso: no se pudo descargar el tipo de cambio de 2024; se usa la copia en tipo_cambio_2024.json.
Aviso: no se pudo descargar el tipo de cambio de 2025; se usa la copia en tipo_cambio_2025.json.
Aviso: no se pudo descargar el tipo de cambio de 2026; se usa la copia en tipo_cambio_2026.json.
Dólar observado diario: 651 días con tasa, desde 2025-01-01 hasta 2026-10-13; 444 fechas publicadas y 207 completadas con el último valor publicado.


In [ ]:
producto_keys = dim_producto[
    ["producto_id", "anada", "producto_anada_key", "litros_por_caja", "costo_caja_clp"]
]
fact = ventas.merge(
    producto_keys,
    on=["producto_id", "anada"],
    how="left",
    validate="many_to_one",
)
fact = fact.merge(
    embarques_fact,
    on="venta_id",
    how="left",
    validate="one_to_one",
)
fact = fact.merge(
    dim_mercado[["mercado_id", "moneda_facturacion"]],
    on="mercado_id",
    how="left",
    validate="many_to_one",
)

fact["dolar_fecha_venta"] = fact["fecha"].map(tasa_por_fecha)
fact["dolar_fecha_embarque"] = fact["fecha_embarque"].map(tasa_por_fecha)
fact["litros"] = fact["cajas"] * fact["litros_por_caja"]
fact["monto_bruto"] = fact["cajas"] * fact["precio_unitario"]
fact["monto_descuento"] = fact["monto_bruto"] - fact["monto_neto"]
fact["ingreso_neto_clp_fecha_venta"] = fact["monto_neto"].where(
    fact["moneda_facturacion"].eq("CLP"),
    fact["monto_neto"] * fact["dolar_fecha_venta"],
)
fact["ingreso_neto_clp_fecha_embarque"] = fact["monto_neto"].where(
    fact["moneda_facturacion"].eq("CLP"),
    fact["monto_neto"] * fact["dolar_fecha_embarque"],
).where(fact["fecha_embarque"].notna())
fact["costo_total_clp"] = fact["cajas"] * fact["costo_caja_clp"]

sin_tasa_venta_usd = fact["moneda_facturacion"].eq("USD") & fact["dolar_fecha_venta"].isna()
sin_tasa_embarque_usd = (
    fact["moneda_facturacion"].eq("USD")
    & fact["fecha_embarque"].notna()
    & fact["dolar_fecha_embarque"].isna()
)
assert not sin_tasa_venta_usd.any(), "Falta tipo de cambio para alguna fecha de venta USD."
assert not sin_tasa_embarque_usd.any(), "Falta tipo de cambio para alguna fecha de embarque USD."
assert fact["moneda_facturacion"].isin(["CLP", "USD"]).all()

columnas_hecho = [
    "venta_id",
    "fecha",
    "mercado_id",
    "cliente_id",
    "producto_anada_key",
    "fecha_embarque",
    "cajas",
    "litros",
    "monto_bruto",
    "monto_descuento",
    "monto_neto",
    "ingreso_neto_clp_fecha_venta",
    "ingreso_neto_clp_fecha_embarque",
    "costo_total_clp",
]
fact_venta = fact[columnas_hecho].rename(
    columns={"monto_neto": "ingreso_neto_moneda_original"}
).copy()

group_currency_id = codigo_por_moneda.reset_index().set_index("moneda")["moneda_id"]
mercado_currency_id = dim_mercado.set_index("mercado_id")["moneda_facturacion"].map(group_currency_id)
assert (
    ventas.dropna(subset=["moneda_id"])["moneda_id"].astype("Int64").reset_index(drop=True)
    == ventas.dropna(subset=["moneda_id"])["mercado_id"]
    .map(mercado_currency_id)
    .astype("Int64")
    .reset_index(drop=True)
).all(), "La moneda limpia de ventas no coincide con la moneda única asignada al mercado."
assert fact_venta["venta_id"].is_unique
assert len(fact_venta) == len(ventas)
assert fact_venta[["fecha", "mercado_id", "cliente_id", "producto_anada_key"]].notna().all().all()
assert (fact_venta["litros"] == fact["cajas"] * fact["litros_por_caja"]).all()

print(f"fact_venta: {len(fact_venta):,} filas; venta_id único: {fact_venta['venta_id'].is_unique}")
print(f"Ventas con fecha de embarque confiable: {fact_venta['fecha_embarque'].notna().sum():,}")
print("Fact table construida y validada; auditoría se ejecuta en el bloque siguiente.")

fact_venta: 13,830 filas; venta_id único: True
Ventas con fecha de embarque confiable: 3,064
Fact table construida y validada; auditoría se ejecuta en el bloque siguiente.


## Auditoría de calidad de ventas

Este bloque se ejecuta por separado del modelado dimensional. Genera una fila por `venta_id` con banderas de calidad y se vincula a `fact_venta` solo mediante ese identificador.

In [41]:
auditoria_venta = ventas[
    ["venta_id", "moneda_imputada", "es_devolucion", "monto_recalculado"]
].rename(
    columns={
        "moneda_imputada": "moneda_inferida",
        "monto_recalculado": "monto_inconsistente",
    }
).merge(
    fact_venta[["venta_id", "fecha_embarque"]]
    .assign(sin_embarque=lambda table: table["fecha_embarque"].isna())
    .drop(columns="fecha_embarque"),
    on="venta_id",
    how="left",
    validate="one_to_one",
)

assert auditoria_venta["venta_id"].is_unique
assert len(auditoria_venta) == len(fact_venta)
assert auditoria_venta["venta_id"].reset_index(drop=True).equals(
    fact_venta["venta_id"].reset_index(drop=True)
)
assert set(auditoria_venta.columns) == {
    "venta_id",
    "moneda_inferida",
    "es_devolucion",
    "monto_inconsistente",
    "sin_embarque",
}

print(f"auditoria_venta: {len(auditoria_venta):,} filas; una por venta_id.")
print(
    f"Banderas: moneda inferida={auditoria_venta['moneda_inferida'].sum():,} | "
    f"devoluciones={auditoria_venta['es_devolucion'].sum():,} | "
    f"monto inconsistente={auditoria_venta['monto_inconsistente'].sum():,} | "
    f"sin embarque={auditoria_venta['sin_embarque'].sum():,}"
)

auditoria_venta: 13,830 filas; una por venta_id.
Banderas: moneda inferida=691 | devoluciones=110 | monto inconsistente=691 | sin embarque=10,766


In [40]:
assert dim_tiempo["fecha"].is_unique
assert dim_mercado["mercado_id"].is_unique
assert dim_cliente["cliente_id"].is_unique
assert dim_producto["producto_anada_key"].is_unique
assert not dim_producto.duplicated(["producto_id", "anada"]).any()
assert mercados_con_varias_monedas.empty
assert dim_mercado["moneda_facturacion"].notna().all()
assert {"litros_por_caja", "costo_caja_clp"}.issubset(dim_producto.columns)
assert fact_venta["fecha"].isin(dim_tiempo["fecha"]).all()
assert fact_venta["mercado_id"].isin(dim_mercado["mercado_id"]).all()
assert fact_venta["cliente_id"].isin(dim_cliente["cliente_id"]).all()
assert fact_venta["producto_anada_key"].isin(dim_producto["producto_anada_key"]).all()
assert fact_venta["fecha_embarque"].dropna().isin(dim_tiempo["fecha"]).all()
assert np.allclose(
    fact_venta["litros"], fact["cajas"] * fact["litros_por_caja"], equal_nan=True
)
assert np.allclose(
    fact_venta["monto_descuento"],
    fact["monto_bruto"] - fact["monto_neto"],
    equal_nan=True,
)
assert np.allclose(
    fact_venta["costo_total_clp"],
    fact["cajas"] * fact["costo_caja_clp"],
    equal_nan=True,
)
assert np.allclose(
    fact_venta.loc[fact["moneda_facturacion"].eq("CLP"), "ingreso_neto_clp_fecha_venta"],
    fact.loc[fact["moneda_facturacion"].eq("CLP"), "monto_neto"],
    equal_nan=True,
)
assert np.allclose(
    fact_venta.loc[fact["moneda_facturacion"].eq("USD"), "ingreso_neto_clp_fecha_venta"],
    fact.loc[fact["moneda_facturacion"].eq("USD"), "monto_neto"]
    * fact.loc[fact["moneda_facturacion"].eq("USD"), "dolar_fecha_venta"],
    equal_nan=True,
)
assert fact_venta.loc[
    fact_venta["fecha_embarque"].isna(), "ingreso_neto_clp_fecha_embarque"
].isna().all()
assert not any(dtype == "object" for dtype in fact_venta.dtypes)
assert all(
    pd.api.types.is_numeric_dtype(fact_venta[columna])
    for columna in fact_venta.columns
    if columna not in {"fecha", "fecha_embarque"}
)
assert not {
    "moneda_facturacion",
    "moneda_id",
    "codigo_moneda",
    "moneda_inferida",
    "es_devolucion",
    "monto_inconsistente",
    "sin_embarque",
    "litros_por_caja",
    "costo_caja_clp",
}.intersection(fact_venta.columns)

print("Integridad del modelo dimensional y medidas de fact_venta: OK")
print("Filas por tabla dimensional y hecho:")
print(
    pd.Series(
        {
            "dim_tiempo": len(dim_tiempo),
            "dim_mercado": len(dim_mercado),
            "dim_cliente": len(dim_cliente),
            "dim_producto": len(dim_producto),
            "fact_venta": len(fact_venta),
        },
        name="filas",
    )
)
print("\nColumnas de fact_venta (venta_id identifica el grano):")
print(fact_venta.columns.tolist())

Integridad del modelo dimensional y medidas de fact_venta: OK
Filas por tabla dimensional y hecho:
dim_tiempo        408
dim_mercado        11
dim_cliente        26
dim_producto       96
fact_venta      13830
Name: filas, dtype: int64

Columnas de fact_venta (venta_id identifica el grano):
['venta_id', 'fecha', 'mercado_id', 'cliente_id', 'producto_anada_key', 'fecha_embarque', 'cajas', 'litros', 'monto_bruto', 'monto_descuento', 'ingreso_neto_moneda_original', 'ingreso_neto_clp_fecha_venta', 'ingreso_neto_clp_fecha_embarque', 'costo_total_clp']


## Ejemplos visuales del modelo

La vista final muestra únicamente `fact_venta` y las dimensiones que la organizan, con sus claves de unión. La auditoría de calidad se ejecuta aparte y no forma parte de estos ejemplos.

In [44]:
from IPython.display import HTML, display

pais_por_venta = ventas["mercado_id"].map(dim_mercado.set_index("mercado_id")["pais"])
ventas_embarcadas = set(embarques_fact["venta_id"])
criterios_ejemplo = [
    pais_por_venta.eq("Chile"),
    pais_por_venta.ne("Chile") & ventas["venta_id"].isin(ventas_embarcadas),
    pais_por_venta.ne("Chile") & ~ventas["venta_id"].isin(ventas_embarcadas),
]

ids_ejemplo = []
for criterio in criterios_ejemplo:
    disponibles = ventas.loc[criterio, "venta_id"].sort_values()
    disponibles = disponibles[~disponibles.isin(ids_ejemplo)]
    if not disponibles.empty:
        ids_ejemplo.append(disponibles.iloc[0])

ids_relleno = ventas.loc[
    ~ventas["venta_id"].isin(ids_ejemplo), "venta_id"
].sort_values().head(max(0, 8 - len(ids_ejemplo)))
ids_ejemplo.extend(ids_relleno.tolist())
muestra_fact = (
    fact_venta[fact_venta["venta_id"].isin(ids_ejemplo)]
    .sort_values("venta_id")
    .copy()
)

fechas_venta_muestra = set(muestra_fact["fecha"])
fechas_embarque_muestra = set(muestra_fact["fecha_embarque"].dropna())
claves_tiempo = fechas_venta_muestra | fechas_embarque_muestra
claves_mercado = muestra_fact["mercado_id"].unique()
claves_cliente = muestra_fact["cliente_id"].unique()
claves_producto = muestra_fact["producto_anada_key"].unique()

dim_tiempo_muestra = dim_tiempo[dim_tiempo["fecha"].isin(claves_tiempo)].copy()
dim_tiempo_visual = dim_tiempo_muestra.copy()
dim_tiempo_visual["uso_en_muestra"] = dim_tiempo_visual["fecha"].map(
    lambda fecha: (
        "Venta y embarque"
        if fecha in fechas_venta_muestra and fecha in fechas_embarque_muestra
        else "Venta"
        if fecha in fechas_venta_muestra
        else "Embarque"
    )
)
dim_mercado_muestra = dim_mercado[dim_mercado["mercado_id"].isin(claves_mercado)].copy()
dim_cliente_muestra = dim_cliente[dim_cliente["cliente_id"].isin(claves_cliente)].copy()
dim_producto_muestra = dim_producto[
    dim_producto["producto_anada_key"].isin(claves_producto)
].copy()

assert muestra_fact["venta_id"].is_unique
assert claves_tiempo.issubset(set(dim_tiempo_muestra["fecha"]))
assert set(claves_mercado).issubset(set(dim_mercado_muestra["mercado_id"]))
assert set(claves_cliente).issubset(set(dim_cliente_muestra["cliente_id"]))
assert set(claves_producto).issubset(set(dim_producto_muestra["producto_anada_key"]))
assert dim_tiempo_muestra["fecha"].is_unique

def tabla_html(frame, fechas=(), numeros=(), montos=()):
    vista = frame.copy()
    for columna in fechas:
        vista[columna] = pd.to_datetime(vista[columna]).dt.strftime("%Y-%m-%d").fillna("-")
    for columna in numeros:
        vista[columna] = vista[columna].map(
            lambda valor: f"{valor:,.1f}" if pd.notna(valor) else "-"
        )
    for columna in montos:
        vista[columna] = vista[columna].map(
            lambda valor: f"{valor:,.0f}" if pd.notna(valor) else "-"
        )
    return vista.to_html(index=False, classes="star-table", border=0)


columnas_hecho = [
    "venta_id",
    "fecha",
    "mercado_id",
    "cliente_id",
    "producto_anada_key",
    "fecha_embarque",
    "cajas",
    "litros",
    "monto_bruto",
    "monto_descuento",
    "ingreso_neto_moneda_original",
    "ingreso_neto_clp_fecha_venta",
    "ingreso_neto_clp_fecha_embarque",
    "costo_total_clp",
]
columnas_monto_hecho = [
    "monto_bruto",
    "monto_descuento",
    "ingreso_neto_moneda_original",
    "ingreso_neto_clp_fecha_venta",
    "ingreso_neto_clp_fecha_embarque",
    "costo_total_clp",
]
html_hecho = tabla_html(
    muestra_fact[columnas_hecho],
    fechas=("fecha", "fecha_embarque"),
    numeros=("cajas", "litros"),
    montos=columnas_monto_hecho,
)

html_dimensiones = [
    (
        "dim_tiempo (una dimensión, dos roles)",
        "fact_venta.fecha y fact_venta.fecha_embarque se relacionan con dim_tiempo.fecha",
        tabla_html(
            dim_tiempo_visual[
                [
                    "fecha",
                    "anio",
                    "mes",
                    "trimestre",
                    "semestre",
                    "estacion_hemisferio_sur",
                    "uso_en_muestra",
                ]
            ],
            fechas=("fecha",),
        ),
    ),
    (
        "dim_mercado",
        "fact_venta.mercado_id = dim_mercado.mercado_id",
        tabla_html(dim_mercado_muestra),
    ),
    (
        "dim_cliente",
        "fact_venta.cliente_id = dim_cliente.cliente_id",
        tabla_html(dim_cliente_muestra),
    ),
    (
        "dim_producto",
        "fact_venta.producto_anada_key = dim_producto.producto_anada_key",
        tabla_html(
            dim_producto_muestra,
            numeros=("litros_por_caja",),
            montos=("costo_caja_clp",),
        ),
    ),
]

bloques_dimensiones = "".join(
    f"""
    <section class="star-dimension">
      <h3>{nombre}</h3>
      <p class="star-link">Unión por clave: <code>{relacion}</code></p>
      <div class="star-table-scroll">{contenido}</div>
    </section>
    """
    for nombre, relacion, contenido in html_dimensiones
)
display(
    HTML(
        """
        <style>
        .star-model { width: 100%; min-width: 0; box-sizing: border-box; color: #23383d; font-family: sans-serif; }
        .star-fact, .star-dimension { border: 1px solid #cad7da; padding: 12px; background: #fff; box-sizing: border-box; }
        .star-fact { width: 100%; min-width: 0; overflow: hidden; border-top: 4px solid #287c75; margin-bottom: 14px; }
        .star-fact .star-table-scroll { width: 100%; min-width: 0; overflow-x: auto; box-sizing: border-box; }
        .star-dimension { border-top: 3px solid #8ba6a3; min-width: 0; }
        .star-model h3 { margin: 0 0 6px; font-size: 15px; }
        .star-model p { margin: 4px 0 10px; font-size: 12px; }
        .star-link code { background: #eef3f3; padding: 2px 4px; }
        .star-table-scroll { max-width: 100%; overflow-x: auto; }
        table.star-table { border-collapse: collapse; width: max-content; min-width: 100%; font-size: 12px; }
        table.star-table th { background: #e8f0f2; color: #18343b; text-align: left; white-space: nowrap; }
        table.star-table th, table.star-table td { padding: 5px 8px; border-bottom: 1px solid #d8e1e3; white-space: nowrap; }
        table.star-table tbody tr:nth-child(even) { background: #f5f8f8; }
        .star-dimensions { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 12px; }
        @media (max-width: 900px) { .star-dimensions { grid-template-columns: 1fr; } }
        </style>
        <div class="star-model">
          <section class="star-fact">
            <h3>Tabla de hechos: fact_venta</h3>
            <p>Una fila por venta_id. Solo claves foráneas y medidas; las dos fechas enlazan con la misma dim_tiempo.</p>
            <div class="star-table-scroll">
        """
        + html_hecho
        + """    </div>
          </section><div class='star-dimensions'>"""
        + bloques_dimensiones
        + "</div></div>"
    )
)
print(
    f"Muestra: {len(muestra_fact)} hechos | "
    f"dim_tiempo única={len(dim_tiempo_muestra)} fechas "
    f"(venta={len(fechas_venta_muestra)}, embarque={len(fechas_embarque_muestra)}) | "
    f"dim_mercado={len(dim_mercado_muestra)}, dim_cliente={len(dim_cliente_muestra)}, "
    f"dim_producto={len(dim_producto_muestra)}"
)

Muestra: 8 hechos | dim_tiempo única=2 fechas (venta=1, embarque=1) | dim_mercado=7, dim_cliente=8, dim_producto=7
